# climagrid Quickstart

This notebook walks through the three ways to use climagrid:

1. **High-level**: `climagrid.run()` in one line
2. **Mid-level**: individual adapters + joiner
3. **Low-level**: fetch raw data, compute features manually

All three paths produce the same output: a Pandas DataFrame with one row per
(asset, hour), containing environmental observations and engineering stress features.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

import climagrid

print(f"climagrid version: {climagrid.__version__}")

/home/devbot/.pyenv/versions/3.12.5/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.2) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


climagrid version: 0.2.0


## 1. High-level: `climagrid.run()`

Pass an asset file and a time range. climagrid fetches the data, joins it to
each asset by nearest-neighbor spatial match, and computes all stress features.

In [2]:
ASSETS = Path("data/sample_assets.csv")   # 33 real substations across 7 U.S. states (OpenStreetMap)
START  = datetime(2024, 7, 15, tzinfo=timezone.utc)
END    = datetime(2024, 7, 15, 6, tzinfo=timezone.utc)  # 6 hours

df = climagrid.run(
    ASSETS,
    start_dt=START,
    end_dt=END,
    sources=["nasa_power"],   # NASA POWER: global, no API key
    features="all",
)

print(f"Shape: {df.shape}")
print(f"Assets: {df['asset_id'].nunique()}")
print(f"Hours:  {df['timestamp'].nunique()}")
df.head(3)

Shape: (792, 19)
Assets: 33
Hours:  24


,asset_id,timestamp,lat,lon,nasa_temperature_2m,nasa_wind_speed_10m,nasa_solar_irradiance_ghi,nasa_relative_humidity_2m,nasa_precipitation,feat_thermal_aging_factor,feat_heat_hours_above_35c,feat_freeze_thaw_cycles,feat_ice_loading_risk,feat_soil_saturation_index,feat_wildfire_proximity,wfigs_nearest_fire_km,wfigs_fire_active,wfigs_fire_area_ha,feat_conductor_sag_index
0,Rudd Substation,2024-07-15 00:00:00+00:00,33.41,-112.3171,42.36,6.99,330.80,17.37,0.0,0.007429,1.0,0.0,0.0,0.0,0.0,inf,False,0.0,0.472320
1,Rudd Substation,2024-07-15 01:00:00+00:00,33.41,-112.3171,41.10,6.96,135.00,18.06,0.0,0.006307,2.0,0.0,0.0,0.0,0.0,inf,False,0.0,0.373172
2,Rudd Substation,2024-07-15 02:00:00+00:00,33.41,-112.3171,38.83,5.46,12.52,20.22,0.0,0.004683,3.0,0.0,0.0,0.0,0.0,inf,False,0.0,0.281958


### What columns are in the output?

In [3]:
# Index columns: always present
print("Index:", [c for c in df.columns if c in {"asset_id", "timestamp", "lat", "lon"}])

# Environmental observations
print("NASA:  ", [c for c in df.columns if c.startswith("nasa_")])

# Stress features
print("Features:", [c for c in df.columns if c.startswith("feat_")])

Index: ['asset_id', 'timestamp', 'lat', 'lon']
NASA:   ['nasa_temperature_2m', 'nasa_wind_speed_10m', 'nasa_solar_irradiance_ghi', 'nasa_relative_humidity_2m', 'nasa_precipitation']
Features: ['feat_thermal_aging_factor', 'feat_heat_hours_above_35c', 'feat_freeze_thaw_cycles', 'feat_ice_loading_risk', 'feat_soil_saturation_index', 'feat_wildfire_proximity', 'feat_conductor_sag_index']


### The thermal aging factor (IEEE C57.91)

`feat_thermal_aging_factor` is the Arrhenius FAA: ratio of insulation aging rate
at observed temperature vs. the 110°C reference. Values > 1 mean accelerated aging.

In [4]:
faa_summary = df.groupby("asset_id")["feat_thermal_aging_factor"].agg(["mean", "max"])
faa_summary.columns = ["mean_FAA", "peak_FAA"]
faa_summary = faa_summary.sort_values("peak_FAA", ascending=False)
print("Top 10 assets by peak thermal aging:")
print(faa_summary.head(10).to_string())

Top 10 assets by peak thermal aging:
                          mean_FAA  peak_FAA
asset_id                                    
Rudd Substation           0.003704  0.007429
Perkins Substation        0.003704  0.007429
Westwing Substation       0.003704  0.007429
Pinnacle Peak Substation  0.003298  0.006645
Kyrene Substation         0.003298  0.006645
Duke Energy Substation    0.002049  0.004995
Durham Substation         0.002049  0.004995
Wake Substation           0.002049  0.004995
Ridge Substation          0.001224  0.003242
Buckley Substation        0.000972  0.002372


## 2. Mid-level: adapter + joiner

Use individual components when you need more control, e.g., to fetch from
multiple sources and merge them yourself.

In [5]:
from climagrid.assets.joiner import AssetEnvironmentJoiner
from climagrid.assets.registry import AssetRegistry
from climagrid.sources.base import BoundingBox
from climagrid.sources.nasa_power import NasaPowerAdapter

# 1. Load assets
registry = AssetRegistry(ASSETS)
print(registry)

# 2. Build a bounding box around central Texas
bbox = BoundingBox(min_lat=31.3, max_lat=31.9, min_lon=-97.4, max_lon=-96.9)
print(f"Bbox center: {bbox.center}")

# 3. Fetch raw data
nasa = NasaPowerAdapter()
raw = nasa.fetch(bbox, START, END)
print(f"Raw data shape: {raw.shape}")
raw.head(3)

AssetRegistry(n=33, path='sample_assets.csv')
Bbox center: (31.6, -97.15)


Raw data shape: (24, 8)


,timestamp,nasa_temperature_2m,nasa_wind_speed_10m,nasa_solar_irradiance_ghi,nasa_relative_humidity_2m,nasa_precipitation,lat,lon
0,2024-07-15 00:00:00+00:00,31.12,5.66,127.12,44.73,0.07,31.6,-97.15
1,2024-07-15 01:00:00+00:00,28.44,4.80,9.30,54.73,0.09,31.6,-97.15
2,2024-07-15 02:00:00+00:00,27.21,4.51,0.00,60.88,0.09,31.6,-97.15


In [6]:
# 4. Join to assets by nearest-neighbor spatial match
joiner = AssetEnvironmentJoiner(max_distance_km=200)
joined = joiner.join(registry, raw)
print(f"Joined shape: {joined.shape}")
joined.head(3)

Joined shape: (792, 9)


/tmp/claude-1000/ipykernel_269242/2116882348.py:3: UserWarning: 33 asset(s) are more than 200 km from any environmental data point. Those rows will have NaN values.
  joined = joiner.join(registry, raw)


,asset_id,timestamp,lat,lon,nasa_temperature_2m,nasa_wind_speed_10m,nasa_solar_irradiance_ghi,nasa_relative_humidity_2m,nasa_precipitation
0,Rudd Substation,2024-07-15 00:00:00+00:00,33.41,-112.3171,NaN,NaN,NaN,NaN,NaN
1,Rudd Substation,2024-07-15 01:00:00+00:00,33.41,-112.3171,NaN,NaN,NaN,NaN,NaN
2,Rudd Substation,2024-07-15 02:00:00+00:00,33.41,-112.3171,NaN,NaN,NaN,NaN,NaN


## 3. Low-level: compute a single feature manually

In [7]:
from climagrid.features.conductor_sag import ConductorSagIndex
from climagrid.features.thermal import ThermalStressIndex

# Add asset_id for grouping (required by ThermalStressIndex)
joined["asset_id"] = joined.get("asset_id", "demo")

# Compute thermal aging: uses nasa_temperature_2m as fallback if hrrr not present
result = ThermalStressIndex().compute(joined)
result = ConductorSagIndex().compute(result)

result[["timestamp", "nasa_temperature_2m",
        "feat_thermal_aging_factor", "feat_conductor_sag_index"]].head(6)

,timestamp,nasa_temperature_2m,feat_thermal_aging_factor,feat_conductor_sag_index
0,2024-07-15 00:00:00+00:00,NaN,NaN,NaN
1,2024-07-15 01:00:00+00:00,NaN,NaN,NaN
2,2024-07-15 02:00:00+00:00,NaN,NaN,NaN
3,2024-07-15 03:00:00+00:00,NaN,NaN,NaN
4,2024-07-15 04:00:00+00:00,NaN,NaN,NaN
5,2024-07-15 05:00:00+00:00,NaN,NaN,NaN


## 4. Export

Wide-form Parquet (one column per feature) is recommended for datasets
larger than 30 days or 100 assets. Long-form is better for ML feature stores.

In [8]:
from climagrid.outputs import to_long_parquet, to_parquet

# Wide-form Parquet
p = to_parquet(df, "/tmp/climagrid_wide.parquet")
print(f"Wide Parquet: {p}  ({p.stat().st_size:,} bytes)")

# Long-form Parquet
p2 = to_long_parquet(df, "/tmp/climagrid_long.parquet")
print(f"Long Parquet: {p2}  ({p2.stat().st_size:,} bytes)")

# Peek at long form
pd.read_parquet(p2).head(6)

Wide Parquet: /tmp/climagrid_wide.parquet  (28,410 bytes)
Long Parquet: /tmp/climagrid_long.parquet  (21,072 bytes)


,asset_id,timestamp,lat,lon,feature_name,feature_value
0,Rudd Substation,2024-07-15 00:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,42.36
1,Rudd Substation,2024-07-15 01:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,41.10
2,Rudd Substation,2024-07-15 02:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,38.83
3,Rudd Substation,2024-07-15 03:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,36.79
4,Rudd Substation,2024-07-15 04:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,35.14
5,Rudd Substation,2024-07-15 05:00:00+00:00,33.41,-112.3171,nasa_temperature_2m,33.73


## 5. Schema reference

In [9]:
climagrid.schema_summary()

,column,dtype,units,source,nullable,description
0,asset_id,str,n/a,index,False,Utility asset identifier from AssetRegistry
1,timestamp,"datetime64[ns, UTC]",n/a,index,False,UTC timestamp of the observation or forecast hour
2,lat,float64,degrees,index,False,Asset latitude (WGS-84)
3,lon,float64,degrees,index,False,Asset longitude (WGS-84)
4,hrrr_temperature_2m,float64,°C,noaa_hrrr,True,2-metre air temperature
5,hrrr_wind_speed_10m,float64,m/s,noaa_hrrr,True,10-metre wind speed (magnitude)
6,hrrr_wind_direction_10m,float64,degrees,noaa_hrrr,True,10-metre wind direction (met convention)
7,hrrr_relative_humidity_2m,float64,%,noaa_hrrr,True,2-metre relative humidity
8,hrrr_precipitation_rate,float64,mm/hr,noaa_hrrr,True,Hourly accumulated precipitation
9,hrrr_solar_irradiance_ghi,float64,W/m²,noaa_hrrr,True,Downward short-wave radiation at surface


## 6. Map: asset thermal stress

Plot each asset location colored by its mean thermal aging factor.
Warmer color = more accelerated insulation aging relative to IEEE C57.91 baseline.
This plot is also saved to `docs/assets/quickstart_map.png` for use in the README.

In [10]:
import matplotlib

matplotlib.use("Agg")
from pathlib import Path

import matplotlib.pyplot as plt

# Reuse df from section 1: every asset already has valid features, because
# climagrid fetches weather at each asset's own location (no single-centroid miss).
asset_summary = (
    df.groupby("asset_id")
    .agg(lat=("lat", "first"), lon=("lon", "first"),
         mean_faa=("feat_thermal_aging_factor", "mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor("#f0f4f8")
fig.patch.set_facecolor("#ffffff")

faa = asset_summary["mean_faa"]
sc = ax.scatter(
    asset_summary["lon"], asset_summary["lat"],
    c=faa, cmap="plasma", vmin=faa.min(), vmax=faa.max(),
    s=160, alpha=1.0, edgecolors="white", linewidths=0.8, zorder=3,
)
cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Mean Thermal Aging Factor (IEEE C57.91)", fontsize=10)
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
n = asset_summary.shape[0]
v = climagrid.__version__
ax.set_title(
    f"Grid Asset Thermal Stress: July 15 2024\n{n} real substations across 7 U.S. states, climagrid {v}",
    fontsize=12, fontweight="bold",
)
ax.grid(True, alpha=0.3, linestyle="--")
ax.tick_params(labelsize=9)
fig.tight_layout()

out = Path("../docs/assets/quickstart_map.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out.resolve()}")
plt.show()

Saved: /home/devbot/Documents/devBOT/career/projects/company-projects/daniel-and-temidire/climagrid/docs/assets/quickstart_map.png


/tmp/claude-1000/ipykernel_269242/1031409277.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
